# Topic: Normalization Techniques (Batch Norm vs. Layer Norm)

## Definition (30-second explanation)
Normalization techniques scale the activations of a neural network layer to have a mean of zero and a variance of one. This stabilizes and accelerates training by smoothing the loss landscape and mitigating "Internal Covariate Shift" (the phenomenon where the distribution of network activations changes as the weights of previous layers are updated during training).

## Why Interviewers Ask This
- To verify you understand *why* deep networks converge (or fail to).
- To test your architectural intuition: knowing exactly why Transformers break if you use Batch Normalization, and why CNNs thrive on it.
- To check if you understand the dimensions/axes over which data is aggregated in different domains (images vs. sequences).

## Core Concepts
- **Internal Covariate Shift (ICS):** As weights change, the inputs to subsequent layers shift wildly. Normalization anchors these distributions.
- **Batch Normalization (BN):** Computes the mean and variance across the *batch* dimension for each individual feature/channel. 
- **Layer Normalization (LN):** Computes the mean and variance across the *feature* dimension for each individual sample in the batch independently.
- **Learnable Parameters:** Both techniques introduce learnable parameters (Gamma $\gamma$ for scaling, Beta $\beta$ for shifting) so the network can undo the normalization if the raw distribution is actually optimal.

## When to Use
- **Batch Normalization:** The standard for Convolutional Neural Networks (CNNs). Applied after the linear/conv transformation and before the activation function.
- **Layer Normalization:** The absolute standard for Recurrent Neural Networks (RNNs), Transformers, and modern LLMs.

## Advantages
- **Batch Norm:** Highly effective at regularizing CNNs; allows for much higher learning rates.
- **Layer Norm:** Independent of batch size. Works perfectly with a batch size of 1, making it ideal for sequence models with variable lengths or memory-constrained LLM training.

## Limitations
- **Batch Norm:** Fails miserably with small batch sizes (estimates of mean/variance become too noisy). Breaks down in sequence models (like NLP) because sentences have different lengths, making batch statistics across sequence steps misaligned.
- **Layer Norm:** Can be computationally heavier per sample than BN in some hardware setups, and doesn't provide the slight regularization effect that the batch-noise of BN provides in image tasks.

## Common Comparisons
- **The Axis of Normalization:** If you have a tensor of shape `(Batch, Features)`:
  - **Batch Norm** normalizes straight down the columns (across the Batch).
  - **Layer Norm** normalizes across the rows (across the Features).

## Common Interview Traps
- **Trap:** Saying Batch Norm is applied *after* the activation function.
  **Fix:** While debated in research, the canonical (and expected interview) answer is that BN is applied *before* the non-linear activation (e.g., Conv2D -> BatchNorm -> ReLU).
- **Trap:** Forgetting how BN behaves in inference.
  **Fix:** State clearly that during inference/testing, BN freezes and uses a running, global moving average of the mean and variance calculated during training. Layer Norm does not need this, as it computes stats on the fly per sample.

## Python / SQL Syntax (if applicable)
    import tensorflow as tf

    # For CNNs (Batch Norm)
    model.add(tf.keras.layers.Conv2D(64, (3, 3)))
    model.add(tf.keras.layers.BatchNormalization())
    model.add(tf.keras.layers.ReLU())

    # For NLP/Transformers (Layer Norm)
    model.add(tf.keras.layers.Dense(512))
    model.add(tf.keras.layers.LayerNormalization(epsilon=1e-6))

## Important Formula (if applicable)
- **Standardization:** $\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$
- **Scale and Shift:** $y = \gamma \hat{x} + \beta$

## 45-Second Interview Answer
"Normalization stabilizes training by keeping layer inputs centered, mitigating internal covariate shift. Batch Normalization normalizes across the batch dimension per feature channel, which is highly efficient for CNNs but requires large batch sizes and struggles with variable-length text sequences. Layer Normalization solves this by normalizing across the features for each individual sample independently. This makes Layer Norm invariant to batch size and the undisputed choice for NLP tasks and Transformer architectures."